In [ ]:
import pandas as pd

# Load datasets
train = pd.read_csv("train.csv")
unlabeled = pd.read_csv("unlabeled.csv")
test = pd.read_csv("test.csv")

In [ ]:
#First five training examples
display(train.head())

,id,text,label
0,1,Cipriani S.A. is an Italian hotel and leisure ...,0
1,2,My Grandpa passed away 3 years ago from cancer...,0
2,3,"""Petit Beurre -- Butter Cookies\n\n1 cup (2 st...",1
3,4,"In an unprecedented move, a dedicated gamer ha...",1
4,5,Chile's copper industry has registered record ...,0


In [ ]:

# Number of rows and columns
print("Dataset shape:", train.shape)

# Human-written and AI-generated passages
print("\nLabel distribution:")
print(train["label"].value_counts())

# Missing values
print("\nMissing values:")
print(train.isnull().sum())


Dataset shape: (10000, 3)

Label distribution:
label
0    5000
1    5000
Name: count, dtype: int64

Missing values:
id       0
text     0
label    0
dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train,
    test_size=2000,
    random_state=42,
    stratify=train["label"]
)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)

Training set: (8000, 3)
Validation set: (2000, 3)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create converter
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

# Learn vocabulary from training data and transform it
X_train = tfidf.fit_transform(train_df["text"])

# Transform validation data
X_val = tfidf.transform(val_df["text"])

# Store the labels
y_train = train_df["label"]
y_val = val_df["label"]

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

Training features: (8000, 10000)
Validation features: (2000, 10000)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import time

lr_model = LogisticRegression(max_iter=1000)

In [ ]:
start_time = time.time()

lr_model.fit(X_train, y_train)

training_time = time.time() - start_time

In [ ]:
# Predict probabilities using validation data
val_probs = lr_model.predict_proba(X_val)[:, 1]

# Validation AUROC
val_auc = roc_auc_score(y_val, val_probs)

# Results
print("Validation AUROC:", round(val_auc, 4))
print("Training time:", round(training_time, 2), "seconds")

Validation AUROC: 0.9222
Training time: 0.12 seconds


In [ ]:
# Predict probabilities using training data
train_probs = lr_model.predict_proba(X_train)[:, 1]

#Training AUROC
train_auc = roc_auc_score(y_train, train_probs)

print("Training AUROC:", round(train_auc, 4))
print("Validation AUROC:", round(val_auc, 4))
print("AUROC gap:", round(train_auc - val_auc, 4))

Training AUROC: 0.9744
Validation AUROC: 0.9222
AUROC gap: 0.0523


In [ ]:
#Naive Baiyes

from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

In [ ]:
start_time = time.time()

nb_model.fit(X_train, y_train)

nb_training_time = time.time() - start_time

In [ ]:
# Probabilities
nb_train_probs = nb_model.predict_proba(X_train)[:, 1]
nb_val_probs = nb_model.predict_proba(X_val)[:, 1]

# AUROC
nb_train_auc = roc_auc_score(y_train, nb_train_probs)
nb_val_auc = roc_auc_score(y_val, nb_val_probs)

# Results
print("Training AUROC:", round(nb_train_auc, 4))
print("Validation AUROC:", round(nb_val_auc, 4))
print("AUROC gap:", round(nb_train_auc - nb_val_auc, 4))
print("Training time:", round(nb_training_time, 2), "seconds")

Training AUROC: 0.9202
Validation AUROC: 0.8654
AUROC gap: 0.0548
Training time: 0.01 seconds


In [ ]:
#SVM

from sklearn.svm import LinearSVC

svm_model = LinearSVC(max_iter=5000, random_state=42)

In [ ]:
start_time = time.time()

svm_model.fit(X_train, y_train)

svm_training_time = time.time() - start_time

In [ ]:
# Decision Scores
svm_train_scores = svm_model.decision_function(X_train)
svm_val_scores = svm_model.decision_function(X_val)

# AUROC
svm_train_auc = roc_auc_score(y_train, svm_train_scores)
svm_val_auc = roc_auc_score(y_val, svm_val_scores)

# Results
print("Training AUROC:", round(svm_train_auc, 4))
print("Validation AUROC:", round(svm_val_auc, 4))
print("AUROC gap:", round(svm_train_auc - svm_val_auc, 4))
print("Training time:", round(svm_training_time, 2), "seconds")

Training AUROC: 0.9997
Validation AUROC: 0.9221
AUROC gap: 0.0776
Training time: 0.15 seconds


In [ ]:
# Hyperparameter tuning for Logistic Regression

c_values = [0.01, 0.1, 1, 10, 100]
lr_results = []

for c in c_values:
    model = LogisticRegression(C=c, max_iter=1000)

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]

    lr_results.append({
        "C": c,
        "Train AUROC": roc_auc_score(y_train, train_probs),
        "Validation AUROC": roc_auc_score(y_val, val_probs),
        "Training Time (s)": training_time
    })


In [ ]:
lr_tuning_df = pd.DataFrame(lr_results)

lr_tuning_df["AUROC Gap"] = (
    lr_tuning_df["Train AUROC"]
    - lr_tuning_df["Validation AUROC"]
)

display(lr_tuning_df.round(4))

best_lr = lr_tuning_df.loc[
    lr_tuning_df["Validation AUROC"].idxmax()
]

print("Best C:", best_lr["C"])
print("Best Validation AUROC:", round(best_lr["Validation AUROC"], 4))


,C,Train AUROC,Validation AUROC,Training Time (s),AUROC Gap
0,0.01,0.8259,0.7979,0.0373,0.0280
1,0.10,0.9061,0.8731,0.0647,0.0330
2,1.00,0.9744,0.9222,0.1084,0.0523
3,10.00,0.9995,0.9246,0.1913,0.0749
4,100.00,1.0000,0.9175,0.2490,0.0825


Best C: 10.0
Best Validation AUROC: 0.9246


In [ ]:
# SVM Hyperparameter Tuning

c_values = [0.01, 0.1, 1, 10, 100]
svm_results = []

for c in c_values:
    model = LinearSVC(C=c, max_iter=5000, random_state=42)

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_scores = model.decision_function(X_train)
    val_scores = model.decision_function(X_val)

    svm_results.append({
        "C": c,
        "Train AUROC": roc_auc_score(y_train, train_scores),
        "Validation AUROC": roc_auc_score(y_val, val_scores),
        "Training Time (s)": training_time
    })


In [ ]:
svm_tuning_df = pd.DataFrame(svm_results)

svm_tuning_df["AUROC Gap"] = (
    svm_tuning_df["Train AUROC"]
    - svm_tuning_df["Validation AUROC"]
)

display(svm_tuning_df.round(4))

best_svm = svm_tuning_df.loc[
    svm_tuning_df["Validation AUROC"].idxmax()
]

print("Best C:", best_svm["C"])
print("Best Validation AUROC:", round(best_svm["Validation AUROC"], 4))


,C,Train AUROC,Validation AUROC,Training Time (s),AUROC Gap
0,0.01,0.9002,0.8680,0.0693,0.0322
1,0.10,0.9757,0.9236,0.0800,0.0522
2,1.00,0.9997,0.9221,0.1939,0.0776
3,10.00,1.0000,0.9019,0.9555,0.0981
4,100.00,1.0000,0.8933,3.1276,0.1067


Best C: 0.1
Best Validation AUROC: 0.9236


In [ ]:
# Training final Logistic Regression model
final_model = LogisticRegression(C=10, max_iter=1000)

# Converting 10,000 passages into TF-IDF features
final_tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_full = final_tfidf.fit_transform(train["text"])

# Training using labeled examples
final_model.fit(X_full, train["label"])

LogisticRegression(C=10, max_iter=1000)

In [ ]:
# Converting test passages
X_test = final_tfidf.transform(test["text"])

# Probability of AI-generated text
test_probs = final_model.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(test_probs))

Number of predictions: 3000


In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "label": test_probs
})

assert len(submission) == 3000
assert submission.columns.tolist() == ["id", "label"]
assert submission["id"].is_unique
assert submission["label"].between(0, 1).all()
assert submission.notnull().all().all()

submission.to_csv("submission.csv", index=False)

display(submission.head())

,id,label
0,13001,0.929243
1,13002,0.039539
2,13003,0.701939
3,13004,0.029858
4,13005,0.973068
